# Práctica 1: BÚSQUEDA (Fase 3: Búsqueda Local)

## Planteamiento
En esta fase se aborda la optimización del recorrido completo para el problema de entregas de última milla. Dado un almacén central y $N$ puntos de entrega alcanzables, el objetivo es encontrar la permutación $\pi$ de orden de visita que minimice la distancia total recorrida $f(\pi)$, regresando finalmente al almacén.

Este problema es una variante del **Problema del Viajante de Comercio (TSP)**, un problema de complejidad **NP-hard**. Para $N > 12$, el espacio de estados $N!$ invalida las búsquedas sistemáticas u óptimas como $A^*$, por lo que se emplean metaheurísticas de **Búsqueda Local**:
1. **Simulated Annealing (SA)** (con enfriamiento geométrico y criterio de Metropolis).
2. **Algoritmo Genético (AG)** (con representación por permutación, cruza Order Crossover - OX y mutación por intercambio).

In [21]:
import sys
from pathlib import Path

# Configurar la ruta raíz del proyecto
RAIZ = Path.cwd().resolve().parent
if str(RAIZ) not in sys.path:
    sys.path.append(str(RAIZ))

import pandas as pd
import matplotlib.pyplot as plt
from src.utils import cargar_grafo, gen_nodo_entrega
from src.fase1 import accesible
from src.fase3 import (
    calcular_matriz_distancias,
    evaluar_costo_ruta,
    simulated_annealing,
    algoritmo_genetico,
    graficar_convergencia,
    graficar_ruta_optimizada
)

print("✓ Módulos importados correctamente.")

✓ Módulos importados correctamente.


## 1. Carga del Grafo Urbano y Selección de Instancias
Cargamos la red vial de las alcaldías Cuauhtémoc, Benito Juárez y Coyoacán descargadas previamente.

In [22]:
from pathlib import Path

# Obtiene la carpeta raíz del proyecto (un nivel arriba de 'notebooks')
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
ruta_grafo = RAIZ / "data" / "cdmx_norte_centro.graphml"

print(f"Buscando archivo en: {ruta_grafo}")

# Cargar el grafo usando la ruta absoluta calculada
G = cargar_grafo(str(ruta_grafo))

Buscando archivo en: C:\Users\Major\Desktop\Inteligencia Artificial\Agente-planificador-de-rutas--main\data\cdmx_norte_centro.graphml


In [23]:
# Generar un almacén y un conjunto de entregas candidato (N=12)
almacen, candidatos = gen_nodo_entrega(G, rango_entregas=(15, 25))
alcanzables, _, _ = accesible(G, almacen, candidatos)

# Seleccionar 12 entregas alcanzables para la prueba
entregas = alcanzables[:12]
puntos = [almacen] + entregas

print(f"Almacén Central: {almacen}")
print(f"Número de Entregas Alcanzables seleccionadas: {len(entregas)}")

Almacén Central: 276312567
Número de Entregas Alcanzables seleccionadas: 12


## 2. Cálculo de la Matriz de Distancias $f(\pi)$
Se precalcula la matriz de distancias más cortas (en metros) entre todos los pares $(u, v)$ de puntos seleccionados mediante los algoritmos de la fase previa.

In [24]:
print("Calculando matriz de distancias punto a punto...")
matriz_dist = calcular_matriz_distancias(G, puntos)

# Evaluar solución trivial aleatoria de referencia
sol_inicial = list(range(len(puntos)))
costo_aleatorio = evaluar_costo_ruta(matriz_dist, sol_inicial)
print(f"Distancia Recorrida con Solución Aleatoria Trivial: {costo_aleatorio:,.2f} metros")

Calculando matriz de distancias punto a punto...
Distancia Recorrida con Solución Aleatoria Trivial: 130,116.87 metros


## 3. Ejecución de Metaheurísticas de Búsqueda Local

### A. Simulated Annealing (SA)
Esquema de enfriamiento geométrico con $T_0 = 1000.0$, $\alpha = 0.95$ y $T_{min} = 0.01$.

In [25]:
print("Ejecutando Simulated Annealing...")
res_sa = simulated_annealing(matriz_dist, T0=1000.0, alpha=0.95, T_min=0.01, max_iter_temp=100)
mejora_sa = ((costo_aleatorio - res_sa['mejor_costo']) / costo_aleatorio) * 100

print(f"  -> Mejor Costo SA : {res_sa['mejor_costo']:,.2f} m")
print(f"  -> Mejora vs Baseline: {mejora_sa:.2f}%")
print(f"  -> Tiempo Cómputo  : {res_sa['tiempo_ms']:.2f} ms")

Ejecutando Simulated Annealing...
  -> Mejor Costo SA : 51,555.32 m
  -> Mejora vs Baseline: 60.38%
  -> Tiempo Cómputo  : 609.37 ms


### B. Algoritmo Genético (AG)
Población de 50 individuos, 250 generaciones, Order Crossover (OX) y mutación por intercambio con $p_m = 0.2$.

In [26]:
print("Ejecutando Algoritmo Genético...")
res_ga = algoritmo_genetico(matriz_dist, num_generaciones=250, tam_poblacion=50, pm=0.2)
mejora_ga = ((costo_aleatorio - res_ga['mejor_costo']) / costo_aleatorio) * 100

print(f"  -> Mejor Costo AG : {res_ga['mejor_costo']:,.2f} m")
print(f"  -> Mejora vs Baseline: {mejora_ga:.2f}%")
print(f"  -> Tiempo Cómputo  : {res_ga['tiempo_ms']:.2f} ms")

Ejecutando Algoritmo Genético...
  -> Mejor Costo AG : 59,379.95 m
  -> Mejora vs Baseline: 54.36%
  -> Tiempo Cómputo  : 1901.16 ms


## 4. Curvas de Convergencia y Visualizaciones de la Ruta Óptima

In [27]:
# Graficar la curva de convergencia
graficar_convergencia(res_sa, res_ga, costo_aleatorio, nombre_archivo="convergencia_notebook.png")

# Graficar la ruta optimizada sobre la red vial real
mejor_res = res_sa if res_sa['mejor_costo'] < res_ga['mejor_costo'] else res_ga
graficar_ruta_optimizada(
    G,
    puntos,
    mejor_res['mejor_ruta'],
    f"Ruta Óptima Fase 3 ({mejor_res['algoritmo']}) - {mejor_res['mejor_costo']:.1f} m",
    nombre_archivo="mapa_notebook.png"
)

  [+] Gráfica de convergencia guardada: convergencia_notebook.png


C:\Users\Major\AppData\Roaming\Python\Python310\site-packages\osmnx\plot.py:351: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with *x* & *y*.  Please use the *color* keyword-argument or provide a 2D array with a single row if you intend to specify the same RGB or RGBA value for all points.
  ax.scatter(od_x, od_y, s=orig_dest_size, c=route_color, alpha=route_alpha, edgecolor="none")


  [+] Mapa de la ruta optimizada guardado: mapa_notebook.png


## 5. Tabla Comparativa de Desempeño y Exportación

In [28]:
df_resultados = pd.DataFrame([
    {
        "Estrategia": "Solución Trivial Aleatoria",
        "Paradas": len(puntos),
        "Costo Total (m)": round(costo_aleatorio, 2),
        "Mejora (%)": "0.00%",
        "Tiempo (ms)": "0.00"
    },
    {
        "Estrategia": "Simulated Annealing (SA)",
        "Paradas": len(puntos),
        "Costo Total (m)": round(res_sa['mejor_costo'], 2),
        "Mejora (%)": f"{mejora_sa:.2f}%",
        "Tiempo (ms)": f"{res_sa['tiempo_ms']:.2f}"
    },
    {
        "Estrategia": "Algoritmo Genético (AG)",
        "Paradas": len(puntos),
        "Costo Total (m)": round(res_ga['mejor_costo'], 2),
        "Mejora (%)": f"{mejora_ga:.2f}%",
        "Tiempo (ms)": f"{res_ga['tiempo_ms']:.2f}"
    }
])

display(df_resultados)

,Estrategia,Paradas,Costo Total (m),Mejora (%),Tiempo (ms)
0,Solución Trivial Aleatoria,13,130116.87,0.00%,0.00
1,Simulated Annealing (SA),13,51555.32,60.38%,609.37
2,Algoritmo Genético (AG),13,59379.95,54.36%,1901.16
